# Weight Initialization in Neural Networks

The standard explanation for why we need to initialize the weights of a neural network properly is based on two factors:

* Symmetry: the network weights must be chosen to ensure that each hidden unit computes a different function. (For a linear layer $\bm W \bm x$, symmetry will be broken if no two rows of $\bm W$ are identical.)
* Vanishing or exploding gradients: if the network weights are too large or too small, the gradients of the loss with respect to the weights will grow or shrink as the gradients are computed backwards through layers.  

#### Imports and setup

In [ ]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [ ]:
import math
from functools import partial

import torch
from torch import nn
import torch.nn.functional as F
from torch.linalg import norm
import matplotlib.pyplot as plt

from utils.data import get_dataloaders
from utils.train import train_model, val_stats
from utils.models import Module

In [ ]:
device='cpu'

## Create model

In [ ]:
def custom_weight_init(init_scale, m):
    if isinstance(m, nn.Linear):
        std = init_scale * (2 / math.sqrt(m.weight.shape[1] + m.weight.shape[0])) # math.sqrt(2 / m.weight.shape[1]) #
        nn.init.normal_(m.weight, std=std)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

def apply_init(model, x, init_scale=1.0):
    model(x) # initialize lazy layers
    init_fn = partial(custom_weight_init, init_scale)
    model.apply(init_fn)

In [ ]:
class MLP(Module):
    def __init__(self, num_layers, num_hiddens, num_classes, use_bias=True, activation=nn.ReLU):
        super().__init__()
        self.layers = nn.Sequential()
        for i in range(num_layers-1):
            self.layers.add_module(f"Dense {i+1}", nn.LazyLinear(num_hiddens, bias=use_bias))
            if activation:
                self.layers.add_module(f"Activation {i+1}", activation())
        self.layers.add_module(f"Linear out (dense {num_layers})", nn.LazyLinear(num_classes, bias=use_bias))
        
    def forward(self, x):
        return self.layers(x)

In [ ]:
dataset = "MNIST" # global variable
num_layers = 5 # global variable

In [ ]:
def get_model(use_bias=True, num_hiddens=64, num_classes=10):
    return MLP(num_layers, num_hiddens, num_classes, use_bias=use_bias, activation=nn.Tanh).to(device)

def init_model(model, init_scale):
    train_dl, _ = get_data()
    for X, _ in train_dl:
        apply_init(model, X, init_scale=init_scale)
        break

loss_fn = lambda X, y: F.cross_entropy(X, y) # F.mse_loss(X, F.one_hot(y).float())

def get_optimizer(model, lr):
    return torch.optim.SGD(model.parameters(), lr=lr)

def get_data(batch_size=128, dataset=dataset):
    preprocess = lambda X, y: (X.flatten(1).to(device), y.to(device))
    return get_dataloaders(batch_size, dataset, func=preprocess)

def gradient_norms(model, X, y, init_scale):
    logits = model(X)
    logits = logits# / init_scale**(num_layers)
    print(f"{norm(logits):.6f}")
    print(f"{logits[0]}")
    #print(f"{torch.softmax(logits, dim=1)[0]}")
    loss = loss_fn(
        logits, y)
    print(f"{loss:.6f}")
    loss.backward()
    weight_grad_norms = []
    bias_grad_norms = []
    for m in model.layers:
        if isinstance(m, nn.Linear):
            weight_grad_norms.append(norm(m.weight.grad).item())
            if m.bias is not None:
                bias_grad_norms.append(norm(m.bias.grad).item())
    return weight_grad_norms, bias_grad_norms

def fit(init_scale=1.0, lr=0.05, use_bias=True, num_epochs=5):
    torch.manual_seed(15)
    model = get_model(use_bias=use_bias)
    init_model(model, init_scale)
    train_dl, val_dl = get_data()
    opt = get_optimizer(model, lr)
    loss = lambda logits, y: loss_fn(logits, y)# / (init_scale)**(num_layers), y)
    train_model(model, train_dl, val_dl, opt, num_epochs, 
                loss_fn=loss, device=device, id=str(init_scale))
    val_loss, val_acc = val_stats(model, val_dl, loss_fn=loss, 
                                  device=device)
    return model, (val_loss, val_acc)

## Weight gradient norms

Plot the L2 norms of the weight gradient matrices as a function of the initialization scale. 

In [ ]:
init_scales = [10**i for i in range(-2, 3)]
print(init_scales)

In [ ]:
weight_grad_norms = []
bias_grad_norms = []


train_dl, _ = get_data(dataset=dataset)

for X, y in train_dl:

    for init_scale in init_scales:
        torch.manual_seed(15)
        model = get_model()
        init_model(model, init_scale)
        wg, bg = gradient_norms(model, X, y, init_scale)
        weight_grad_norms.append(wg)
        bias_grad_norms.append(bg)

        print(f"Init scale: {init_scale}\n\t",end="")
        for grad_norm in weight_grad_norms[-1]:
            print(f"{grad_norm:.4e}", end=" ")
        print("\n\t",end="")
        for bias_norm in bias_grad_norms[-1]:
            print(f"{bias_norm:.4e}", end=" ")
        print("\n",end="")

        print(f"Norm of first weight matrix: {norm(model.layers[-1].weight)}")
    
    break

In [ ]:
def plot_grad_norms(norms, name):
    x = torch.arange(len(norms[0])) + 1
    labels = init_scales[::-1]
    plt.plot(x, torch.tensor(norms).T.flip(1), label=labels)
    plt.yscale('log')
    plt.title(f"{name} norm in 5-layer ReLU MLP")
    plt.xlabel("Linear layer number")
    plt.ylabel(f"L2 Norm of {name} gradient")
    plt.xticks(list(range(1,5+1)))
    plt.legend(title="Init scale")
    plt.show()

In [ ]:
plot_grad_norms(weight_grad_norms, name="weight")

As we can see, the norm of the weight gradients in a ReLU MLP is constant across the network. 

By choosing the learning rate carefully, we can ensure the the magnitude of the updates to the weight matrices $\frac{||\Delta W||}{||W||}$ doesn't depend on the initialization scale.

In [ ]:
plot_grad_norms(bias_grad_norms, name="bias")

However, the norm of the bias vector gradients is not constant across the network. This means that with a constant per-parameter learning rate (i.e., SGD), the magnitude (i.e., norm) of updates to the bias vectors will vary by orders of magnitude across the network. 

Unfortunately, this means that we cannot choose the learning rate carefully to ensure that the magnitude of the updates to the bias matrix $\frac{||\Delta b||}{||b||}$ doesn't depend on the init_scale.

This is the reason that we need to initialize a ReLU MLP to the correct scale: if the scale of the initialization is not correct, the bias gradient will explode or vanish across the network.

# Confirming our intuition experimentally

We will now experimentally verify that ReLU MLPs without proper initialization cannot be trained to the same accuracy as those with good initialization, even if the learning rate is tuned. 

## Choosing the learning rate

We just discovered that the bias vector gradient norm will not be constant across a ReLU MLP. This means that no matter what learning rate we choose, the norm of the bias update will explode or vanish across the network. 

Neverthless, we choose a learning rate that gives a scaled weight update norm $\frac{||\Delta W||}{||W||}$ that doesn't depend on the initialization scale.

In [ ]:
default_lr = 0.5#0.05
init_scales = [10**i for i in range(-2, 3)]
print(init_scales)
learning_rates = [default_lr * (10**2)**i for i in range(-2, 3)] # [50000.0, 50.0, 0.05, 5e-10, 5e-18]
print(learning_rates)

## Training the models

In [ ]:
val_losses = []
val_accuracies = []

for lr, init_scale in zip(learning_rates, init_scales):
    torch.manual_seed(15)
    _, (val_loss, val_acc) = fit(init_scale=init_scale, lr=lr, use_bias=False)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

plt.show()

In [ ]:
print(val_losses)
print(val_accuracies)